In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1/config.json
/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1/training_args.bin
/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1/tokenizer.json
/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1/tokenizer_config.json
/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1/model.safetensors
/kaggle/input/models/alikakaylanisha/indobert-base-p1-toxic-span/transformers/default/1/config.json
/kaggle/input/models/alikakaylanisha/indobert-base-p1-toxic-span/transformers/default/1/training_args.bin
/kaggle/input/models/alikakaylanisha/indobert-base-p1-toxic-span/transformers/default/1/tokenizer.json
/kaggle/input/models/alikakaylanisha/indobert-base-p1-toxic-span/transformers/default/1/tokenizer_config.json
/kaggle/input/models/alikakayla

In [2]:
!pip install datasets
!pip install transformers
!pip install evaluate
!pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=bb0ac5737b19ae762e8dd00700d3d1123a06bc05c92e5c1c366b8668f2f70e61
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
print(pd.__version__)

2.3.3


## load model

In [4]:
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## load dataset

In [5]:
label_list = ["O", "B-TOXIC", "I-TOXIC"]
label2id = {"O": 0, "B-TOXIC": 1, "I-TOXIC": 2}
id2label = {0: "O", 1: "B-TOXIC", 2: "I-TOXIC"}

In [6]:
import json

with open("/kaggle/input/datasets/alikakaylanisha/toxic-span-id-gamechat-4-0/ALL_IOB_1.0.jsonl") as user_file:
  file_contents = user_file.read()


In [7]:
from datasets import load_dataset
file_path = "/kaggle/input/datasets/alikakaylanisha/toxic-span-id-gamechat-4-0/ALL_IOB__encoded_1.0.jsonl"
dataset = load_dataset("json", data_files=file_path)

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
dataset['train']

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 5385
})

In [9]:
from datasets import DatasetDict

dataset_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 4308
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1077
    })
})


In [10]:
train_dataset = dataset_split["train"]
test_val_dataset = dataset_split["test"]

In [11]:
test_val_split = test_val_dataset.train_test_split(test_size= 0.5, seed=42)
test_dataset = test_val_split["train"]
val_dataset = test_val_split["test"]

In [12]:
test_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 538
})

In [13]:
val_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 539
})

In [14]:
train_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 4308
})

In [15]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, return_offsets_mapping=True)

    labels = []
    #addition
    all_word_ids = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
                
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
                
            else:
                label_ids.append(-100)
                
            previous_word_idx = word_idx
            
        labels.append(label_ids)
        
        # SAVE word_ids (addition)
        all_word_ids.append([
            -1 if w is None else w
            for w in word_ids
        ])
        
    tokenized_inputs["word_ids"] = all_word_ids
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [16]:
tokenized_dataset_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_val = val_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_test = test_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4308 [00:00<?, ? examples/s]

Map:   0%|          | 0/539 [00:00<?, ? examples/s]

Map:   0%|          | 0/538 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset_train

Dataset({
    features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
    num_rows: 4308
})

In [18]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# indobenchmark/indobert-base-p1

## Testing

In [19]:
tokenized_dataset_test[0]

{'tokens': ['bencong', 'rombengmenjijikangay', 'jg'],
 'ner_tags': [1, 2, 0],
 'input_ids': [2,
  358,
  27921,
  3610,
  1771,
  5067,
  2455,
  11084,
  24587,
  138,
  6162,
  3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'offset_mapping': [[0, 0],
  [0, 3],
  [3, 7],
  [0, 3],
  [3, 6],
  [6, 8],
  [8, 11],
  [11, 14],
  [14, 18],
  [18, 20],
  [0, 2],
  [0, 0]],
 'word_ids': [-1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, -1],
 'labels': [-100, 1, -100, 2, -100, -100, -100, -100, -100, -100, 0, -100]}

In [20]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "/kaggle/input/models/alikakaylanisha/indobert-base-p1-toxic-span/transformers/default/1"

model = AutoModelForTokenClassification.from_pretrained(model_path)

print("IndoBERT model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IndoBERT model and tokenizer loaded successfully!


In [21]:
import numpy as np
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
)

In [22]:
output = trainer.predict(tokenized_dataset_test)

predictions = output.predictions
labels = output.label_ids
pred_ids = np.argmax(predictions, axis=2)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [23]:
output.metrics

{'test_loss': 0.3136853873729706,
 'test_runtime': 4.0451,
 'test_samples_per_second': 133.002,
 'test_steps_per_second': 8.405}

In [24]:
print(pred_ids.shape)
print(labels.shape)

(538, 166)
(538, 166)


In [25]:
tokenized_dataset_test[0]

{'tokens': ['bencong', 'rombengmenjijikangay', 'jg'],
 'ner_tags': [1, 2, 0],
 'input_ids': [2,
  358,
  27921,
  3610,
  1771,
  5067,
  2455,
  11084,
  24587,
  138,
  6162,
  3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'offset_mapping': [[0, 0],
  [0, 3],
  [3, 7],
  [0, 3],
  [3, 6],
  [6, 8],
  [8, 11],
  [11, 14],
  [14, 18],
  [18, 20],
  [0, 2],
  [0, 0]],
 'word_ids': [-1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, -1],
 'labels': [-100, 1, -100, 2, -100, -100, -100, -100, -100, -100, 0, -100]}

In [26]:
predictions.shape

(538, 166, 3)

### prediction reverse engineering

In [27]:
import numpy as np

def reconstruct_word_labels(pred_ids, word_ids, id2label):

    word_labels = []

    previous_word_id = None

    for pred_id, word_id in zip(pred_ids, word_ids):

        # Skip special tokens
        if word_id == -1:
            continue

        # First subtoken only
        if word_id != previous_word_id:

            word_labels.append(id2label[pred_id])

        previous_word_id = word_id

    return word_labels


In [28]:
def word_labels_to_char_spans(words, word_labels):

    text = " ".join(words)
    spans = []

    current_pos = 0

    for word, label in zip(words, word_labels):

        start = current_pos
        end = start + len(word)

        if label.startswith("B") or label.startswith("I"):
            spans.extend(range(start, end))

        current_pos = end + 1

    return text, spans

In [29]:
tokenized_dataset_test[0]

{'tokens': ['bencong', 'rombengmenjijikangay', 'jg'],
 'ner_tags': [1, 2, 0],
 'input_ids': [2,
  358,
  27921,
  3610,
  1771,
  5067,
  2455,
  11084,
  24587,
  138,
  6162,
  3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'offset_mapping': [[0, 0],
  [0, 3],
  [3, 7],
  [0, 3],
  [3, 6],
  [6, 8],
  [8, 11],
  [11, 14],
  [14, 18],
  [18, 20],
  [0, 2],
  [0, 0]],
 'word_ids': [-1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, -1],
 'labels': [-100, 1, -100, 2, -100, -100, -100, -100, -100, -100, 0, -100]}

In [30]:
pred_char_spans = []
pred_tokens = []
tokens_p = []
sentences_p = []

for pred_seq, sample in zip(pred_ids, tokenized_dataset_test):
  
    word_level_preds = reconstruct_word_labels(
        pred_seq,
        sample["word_ids"],
        id2label
    )
    

    sentence, char_spans = word_labels_to_char_spans(
    sample["tokens"],
    word_level_preds
    )

    pred_char_spans.append(char_spans)
    sentences_p.append(sentence)
    pred_tokens.append(word_level_preds)
    tokens_p.append(sample["tokens"])

In [31]:
predictions_dict = {
    "sentence_p" : sentences_p,
    "token_p" : tokens_p,
    "pred_char_spans" : pred_char_spans,
    "pred_tokens" : pred_tokens
}

In [32]:
tokenized_dataset_test[0]

{'tokens': ['bencong', 'rombengmenjijikangay', 'jg'],
 'ner_tags': [1, 2, 0],
 'input_ids': [2,
  358,
  27921,
  3610,
  1771,
  5067,
  2455,
  11084,
  24587,
  138,
  6162,
  3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'offset_mapping': [[0, 0],
  [0, 3],
  [3, 7],
  [0, 3],
  [3, 6],
  [6, 8],
  [8, 11],
  [11, 14],
  [14, 18],
  [18, 20],
  [0, 2],
  [0, 0]],
 'word_ids': [-1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 2, -1],
 'labels': [-100, 1, -100, 2, -100, -100, -100, -100, -100, -100, 0, -100]}

In [33]:
gold_char_spans = []
gold_tokens = []
tokens_g = []
sentences_g = []

for i in tokenized_dataset_test:
    char_span = []
    sentence = " ".join(i["tokens"])
    for token, tag in zip(i["tokens"], i["ner_tags"]):
    
        if tag !=0: #if tags not O or 0
            start = sentence.find(token)
            end = start + len(token)
            idx = list(range(start, end))
            char_span.extend(idx)
            
        
    gold_char_spans.append(char_span)
    gold_tokens.append(i["ner_tags"])
    tokens_g.append(i["tokens"])
    sentences_g.append(sentence)

In [34]:
gold_tokens[0]

[1, 2, 0]

In [35]:
tokens_g[0]

['bencong', 'rombengmenjijikangay', 'jg']

In [36]:
gold_dict = {
    "sentence_g" : sentences_g,
    "token_g" : tokens_g,
    "gold_char_spans" : gold_char_spans,
    "gold_tokens" : gold_tokens
}

In [37]:
pred_ids.shape

(538, 166)

In [38]:
def decode_valid_tokens(pred_seq, gold_seq, id2label):

    pred_labels = []
    gold_labels = []

    for pred_id, gold_id in zip(pred_seq, gold_seq):

        # ignore subwords
        if gold_id == -100:
            continue

        pred_labels.append(id2label[int(pred_id)])
        gold_labels.append(id2label[int(gold_id)])

    return pred_labels, gold_labels

In [39]:
def classify_match(gold_labels, pred_labels):
    if pred_labels == gold_labels:
        return "exact"
    has_partial_overlap = False
    
    for gold, pred in zip(gold_labels, pred_labels):
        if gold != "O" and gold == pred:
            has_partial_overlap = True
            break  

    return "partial" if has_partial_overlap else "none"

In [40]:
results = []

for pred_seq, gold_seq, sample in zip(
    pred_ids,
    tokenized_dataset_test["labels"],
    tokenized_dataset_test):

    pred_labels, gold_labels = decode_valid_tokens(
        pred_seq,
        gold_seq,
        id2label
    )
    print(gold_labels)
    print(pred_labels)

    match_type = classify_match(
        gold_labels,
        pred_labels
    )

    results.append({

        "text": " ".join(sample["tokens"]),

        "gold": gold_labels,

        "pred": pred_labels,

        "match_type": match_type
    })

['B-TOXIC', 'I-TOXIC', 'O']
['B-TOXIC', 'I-TOXIC', 'O']
['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'B-TOXIC']
['O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'B-TOXIC']
['O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC']
['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', '

In [41]:
exact_matches = [x for x in results if x["match_type"] == "exact"]

partial_matches = [x for x in results if x["match_type"] == "partial"]

no_matches = [x for x in results if x["match_type"] == "none"]

In [42]:
partial_matches[0]

{'text': 'user perbuatan kaum nasrani yahudi kali om utk memecah belah umat muslim astagfirullah',
 'gold': ['O',
  'O',
  'O',
  'B-TOXIC',
  'I-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'pred': ['O',
  'O',
  'O',
  'O',
  'I-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'match_type': 'partial'}

In [43]:
print("Exact :", len(exact_matches))
print("Partial :", len(partial_matches))
print("None :", len(no_matches))

Exact : 367
Partial : 104
None : 67


In [44]:
def print_examples(match_group, n=5):

    for sample in match_group[:n]:

        print("TEXT  :", sample["text"])

        print("GOLD  :", sample["gold"])

        print("PRED  :", sample["pred"])

        print("MATCH :", sample["match_type"])

        print("-" * 60)

In [45]:
print_examples(exact_matches)

TEXT  : bencong rombengmenjijikangay jg
GOLD  : ['B-TOXIC', 'I-TOXIC', 'O']
PRED  : ['B-TOXIC', 'I-TOXIC', 'O']
MATCH : exact
------------------------------------------------------------
TEXT  : user user user juara ngibul junjungan ban kampret
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
MATCH : exact
------------------------------------------------------------
TEXT  : eh adaaa cek toko sebelah \xf0\x9f\x98\x81 sorry yaa inget nonton film klasik indo tokohnya kristen
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : exact
------------------------------------------------------------
TEXT  : emang ya kesalahan udah budaya apaapa dibodoamatin udh yg haha semoga sukses jalannya masingmasing ya
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'O', '

In [46]:
print_examples(no_matches)

TEXT  : aaduh min kelihatan tua ya bliat deh kulit becky 30an menjelang 40 agua nanya kulit guabukan becky bego
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-TOXIC']
MATCH : none
------------------------------------------------------------
TEXT  : apasal laa rindu eeeee idiot laa \xf0\x9f\x98\xad
GOLD  : ['O', 'O', 'O', 'O', 'B-TOXIC', 'O', 'O']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : none
------------------------------------------------------------
TEXT  : user manis cuman butut goblog
GOLD  : ['O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'I-TOXIC']
MATCH : none
------------------------------------------------------------
TEXT  : anjirrrr pa banget bro ngeluarin ultinya berbalik ni
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : none
--------

In [47]:
print_examples(partial_matches)

TEXT  : user perbuatan kaum nasrani yahudi kali om utk memecah belah umat muslim astagfirullah
GOLD  : ['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'O', 'O', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : partial
------------------------------------------------------------
TEXT  : user user user tolol die gaada sangku pautnya ngapain lu blg pengecutdasar tolol
GOLD  : ['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'B-TOXIC']
MATCH : partial
------------------------------------------------------------
TEXT  : user user anjir ga gitu jg bloon
GOLD  : ['O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
MATCH : partial
------------------------------------------------------------
TEXT  : ku genjot kontol kluar masuk memek kencang tangan remas tetek pilin puting susu sssstt

{'text': 'bencong rombengmenjijikangay jg',
 'gold': ['B-TOXIC', 'I-TOXIC', 'O'],
 'pred': ['B-TOXIC', 'O', 'O'],
 'match_type': 'partial'}

### saving prediction results

In [48]:
import csv

# CSV file name
csv_filename = "partial_indobert-base-p1.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  # Write header row
    writer.writerows(partial_matches)  # Write data rows

In [49]:
# CSV file name
csv_filename = "exact_indobert-base-p1.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  
    writer.writerows(exact_matches) # Write data rows

In [50]:
# CSV file name
csv_filename = "none_indobert-base-p1.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  
    writer.writerows(no_matches)  

# indolem/indobert-base-uncased

## load model

In [51]:
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("indolem/indobert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [52]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, return_offsets_mapping=True)

    labels = []
    #addition
    all_word_ids = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
                
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
                
            else:
                label_ids.append(-100)
                
            previous_word_idx = word_idx
            
        labels.append(label_ids)
        
        # SAVE word_ids (addition)
        all_word_ids.append([
            -1 if w is None else w
            for w in word_ids
        ])
        
    tokenized_inputs["word_ids"] = all_word_ids
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [53]:
tokenized_dataset_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_val = val_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset_test = test_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4308 [00:00<?, ? examples/s]

Map:   0%|          | 0/539 [00:00<?, ? examples/s]

Map:   0%|          | 0/538 [00:00<?, ? examples/s]

In [54]:
tokenized_dataset_train

Dataset({
    features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'word_ids', 'labels'],
    num_rows: 4308
})

In [55]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Testing

In [56]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "/kaggle/input/models/alikakaylanisha/indobert-base-uncased-toxic-span/transformers/default/1"

model = AutoModelForTokenClassification.from_pretrained(model_path)

print("IndoBERT model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IndoBERT model and tokenizer loaded successfully!


In [57]:
import numpy as np
from transformers import Trainer


trainer = Trainer(
    model=model,
    data_collator=data_collator,
)

In [58]:
output = trainer.predict(tokenized_dataset_test)

predictions = output.predictions
labels = output.label_ids
pred_ids = np.argmax(predictions, axis=2)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [59]:
output.metrics

{'test_loss': 0.4291333556175232,
 'test_runtime': 2.4884,
 'test_samples_per_second': 216.203,
 'test_steps_per_second': 13.663}

In [60]:
print(pred_ids.shape)
print(labels.shape)

(538, 164)
(538, 164)


In [61]:
tokenized_dataset_test[0]

{'tokens': ['bencong', 'rombengmenjijikangay', 'jg'],
 'ner_tags': [1, 2, 0],
 'input_ids': [3,
  1828,
  22331,
  2977,
  14058,
  28271,
  1478,
  4190,
  11028,
  1534,
  950,
  23599,
  4],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'offset_mapping': [[0, 0],
  [0, 3],
  [3, 7],
  [0, 3],
  [3, 6],
  [6, 8],
  [8, 10],
  [10, 12],
  [12, 17],
  [17, 19],
  [19, 20],
  [0, 2],
  [0, 0]],
 'word_ids': [-1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, -1],
 'labels': [-100,
  1,
  -100,
  2,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  0,
  -100]}

In [62]:
predictions.shape

(538, 164, 3)

### prediction reverse engineering

In [63]:
import numpy as np

def reconstruct_word_labels(pred_ids, word_ids, id2label):

    word_labels = []

    previous_word_id = None

    for pred_id, word_id in zip(pred_ids, word_ids):

        # Skip special tokens
        if word_id == -1:
            continue

        # First subtoken only
        if word_id != previous_word_id:

            word_labels.append(id2label[pred_id])

        previous_word_id = word_id

    return word_labels


In [64]:
def word_labels_to_char_spans(words, word_labels):

    text = " ".join(words)
    spans = []

    current_pos = 0

    for word, label in zip(words, word_labels):

        start = current_pos
        end = start + len(word)

        if label.startswith("B") or label.startswith("I"):
            spans.extend(range(start, end))

        current_pos = end + 1

    return text, spans

In [65]:
pred_char_spans = []
pred_tokens = []
tokens_p = []
sentences_p = []

for pred_seq, sample in zip(pred_ids, tokenized_dataset_test):
  
    word_level_preds = reconstruct_word_labels(
        pred_seq,
        sample["word_ids"],
        id2label
    )
    

    sentence, char_spans = word_labels_to_char_spans(
    sample["tokens"],
    word_level_preds
    )

    pred_char_spans.append(char_spans)
    sentences_p.append(sentence)
    pred_tokens.append(word_level_preds)
    tokens_p.append(sample["tokens"])

In [66]:
predictions_dict = {
    "sentence_p" : sentences_p,
    "token_p" : tokens_p,
    "pred_char_spans" : pred_char_spans,
    "pred_tokens" : pred_tokens
}

In [67]:
gold_char_spans = []
gold_tokens = []
tokens_g = []
sentences_g = []

for i in tokenized_dataset_test:
    char_span = []
    sentence = " ".join(i["tokens"])
    for token, tag in zip(i["tokens"], i["ner_tags"]):
    
        if tag !=0: #if tags not O or 0
            start = sentence.find(token)
            end = start + len(token)
            idx = list(range(start, end))
            char_span.extend(idx)
            
        
    gold_char_spans.append(char_span)
    gold_tokens.append(i["ner_tags"])
    tokens_g.append(i["tokens"])
    sentences_g.append(sentence)

In [68]:
gold_tokens[0]

[1, 2, 0]

In [69]:
tokens_g[0]

['bencong', 'rombengmenjijikangay', 'jg']

In [70]:
gold_dict = {
    "sentence_g" : sentences_g,
    "token_g" : tokens_g,
    "gold_char_spans" : gold_char_spans,
    "gold_tokens" : gold_tokens
}

In [71]:
pred_ids.shape

(538, 164)

In [72]:
def decode_valid_tokens(pred_seq, gold_seq, id2label):

    pred_labels = []
    gold_labels = []

    for pred_id, gold_id in zip(pred_seq, gold_seq):

        # ignore subwords
        if gold_id == -100:
            continue

        pred_labels.append(id2label[int(pred_id)])
        gold_labels.append(id2label[int(gold_id)])

    return pred_labels, gold_labels

In [73]:
def classify_match(gold_labels, pred_labels):
    if pred_labels == gold_labels:
        return "exact"
    has_partial_overlap = False
    
    for gold, pred in zip(gold_labels, pred_labels):
        if gold != "O" and gold == pred:
            has_partial_overlap = True
            break  

    return "partial" if has_partial_overlap else "none"

In [74]:
results = []

for pred_seq, gold_seq, sample in zip(
    pred_ids,
    tokenized_dataset_test["labels"],
    tokenized_dataset_test):

    pred_labels, gold_labels = decode_valid_tokens(
        pred_seq,
        gold_seq,
        id2label
    )

    match_type = classify_match(
        gold_labels,
        pred_labels
    )

    results.append({

        "text": " ".join(sample["tokens"]),

        "gold": gold_labels,

        "pred": pred_labels,

        "match_type": match_type
    })

In [75]:
exact_matches = [x for x in results if x["match_type"] == "exact"]

partial_matches = [x for x in results if x["match_type"] == "partial"]

no_matches = [x for x in results if x["match_type"] == "none"]

In [76]:
partial_matches[0]

{'text': 'user user user tolol die gaada sangku pautnya ngapain lu blg pengecutdasar tolol',
 'gold': ['O',
  'O',
  'O',
  'B-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-TOXIC'],
 'pred': ['O',
  'O',
  'O',
  'B-TOXIC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-TOXIC',
  'I-TOXIC'],
 'match_type': 'partial'}

In [77]:
print("Exact :", len(exact_matches))
print("Partial :", len(partial_matches))
print("None :", len(no_matches))

Exact : 305
Partial : 124
None : 109


In [78]:
def print_examples(match_group, n=5):

    for sample in match_group[:n]:

        print("TEXT  :", sample["text"])

        print("GOLD  :", sample["gold"])

        print("PRED  :", sample["pred"])

        print("MATCH :", sample["match_type"])

        print("-" * 60)

In [79]:
print_examples(exact_matches)

TEXT  : bencong rombengmenjijikangay jg
GOLD  : ['B-TOXIC', 'I-TOXIC', 'O']
PRED  : ['B-TOXIC', 'I-TOXIC', 'O']
MATCH : exact
------------------------------------------------------------
TEXT  : user perbuatan kaum nasrani yahudi kali om utk memecah belah umat muslim astagfirullah
GOLD  : ['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : exact
------------------------------------------------------------
TEXT  : user user user juara ngibul junjungan ban kampret
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
MATCH : exact
------------------------------------------------------------
TEXT  : eh adaaa cek toko sebelah \xf0\x9f\x98\x81 sorry yaa inget nonton film klasik indo tokohnya kristen
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O

In [80]:
print_examples(no_matches)

TEXT  : user user emang mah gitu terkutuk bry
GOLD  : ['O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'O']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : none
------------------------------------------------------------
TEXT  : aaduh min kelihatan tua ya bliat deh kulit becky 30an menjelang 40 agua nanya kulit guabukan becky bego
GOLD  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : none
------------------------------------------------------------
TEXT  : user tiati mencret \xf0\x9f\x98\x91\xf0\x9f\x98\x92
GOLD  : ['O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'I-TOXIC', 'O']
MATCH : none
------------------------------------------------------------
TEXT  : user anak pecun ga jauh2 dr hobi zina haha
GOLD  : ['O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O']
PRED  : ['O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O']
MATCH : none
--------

In [81]:
print_examples(partial_matches)

TEXT  : user user user tolol die gaada sangku pautnya ngapain lu blg pengecutdasar tolol
GOLD  : ['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-TOXIC', 'I-TOXIC']
MATCH : partial
------------------------------------------------------------
TEXT  : user user anjir ga gitu jg bloon
GOLD  : ['O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'B-TOXIC']
PRED  : ['O', 'O', 'B-TOXIC', 'O', 'O', 'O', 'O']
MATCH : partial
------------------------------------------------------------
TEXT  : ku genjot kontol kluar masuk memek kencang tangan remas tetek pilin puting susu ssssttt aaaahhh
GOLD  : ['O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'B-TOXIC', 'O', 'O', 'B-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'O', 'O']
PRED  : ['O', 'B-TOXIC', 'I-TOXIC', 'O', 'O', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'I-TOXIC', 'O']
MATCH : partial
------------------------------

{'text': 'bencong rombengmenjijikangay jg',
 'gold': ['B-TOXIC', 'I-TOXIC', 'O'],
 'pred': ['B-TOXIC', 'O', 'O'],
 'match_type': 'partial'}

### saving prediction results

In [82]:
import csv

# CSV file name
csv_filename = "partial_indobert-base-uncased.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  
    writer.writerows(partial_matches)  

In [83]:
# CSV file name
csv_filename = "exact_indobert-base-uncased.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  
    writer.writerows(exact_matches) 

In [84]:
# CSV file name
csv_filename = "none_indobert-base-uncased.csv"

# field
fieldnames = ["text", "gold", "pred", "match_type"]

# write to CSV
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  
    writer.writerows(no_matches) 